___
# <center>Associação entre duas variáveis numéricas</center>
___

## Aula 18

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * fazer o gráfico de dispersão antes de calcular qualquer número;
 * calcular e interpretar o coeficiente de correlação, sabendo o que ele esconde;
 * ajustar uma regressão linear simples e interpretar o coeficiente na unidade
   das variáveis;
 * distinguir associação de causa, com um exemplo da própria base.

___
<div id="indice"></div>

## Índice

- [Gráfico de dispersão](#dispersao)
- [Correlação](#correlacao)
- [Regressão linear simples](#regressao)
- [Correlação não é causa](#causa)
- [Exercício](#exercicio)

> **Sobre a base.** Os dados são do projeto da clínica com o BTG e estão cobertos
> por sigilo bancário e pelo termo de parceria. O arquivo `btg_aulas.csv` está no
> BlackBoard e **não pode ser compartilhado** fora da turma, nem subido para o
> GitHub, nem colado em ferramentas externas além das usadas em aula.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_aulas.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
processos = pd.read_csv("btg_aulas.csv")
processos.shape

[Volta ao Índice](#indice)

___
<div id="dispersao"></div>

# Gráfico de dispersão

Pergunta: **nos processos em que o banco pagou, o valor pago acompanha o valor
financiado?** As duas variáveis são numéricas e vêm da planilha do banco.

In [ ]:
pagos = processos[processos["pagou"]].dropna(subset=["valor_pago", "valor_financiado"])
len(pagos)

In [ ]:
from plotnine import ggplot, aes, geom_point, geom_smooth, labs, theme_minimal

(ggplot(pagos, aes(x="valor_financiado", y="valor_pago"))
 + geom_point(alpha=0.6, color="#1f6fb4")
 + labs(x="valor financiado (R$)", y="valor pago no processo (R$)")
 + theme_minimal())

[Volta ao Índice](#indice)

___
<div id="correlacao"></div>

# Correlação

O coeficiente de correlação de Pearson mede o quanto as duas variáveis andam
juntas **em linha reta**, numa escala de -1 a 1. Ele não tem unidade.

In [ ]:
pagos[["valor_financiado", "valor_pago"]].corr().round(3)

r = 0,34: positiva e fraca a moderada. O número sozinho esconde o formato da
nuvem. Quatro conjuntos de dados podem ter a mesma correlação e formatos
completamente diferentes (o quarteto de Anscombe, nos slides). **Olhe sempre o
gráfico.**

[Volta ao Índice](#indice)

___
<div id="regressao"></div>

# Regressão linear simples

A regressão escolhe a reta que torna a soma dos quadrados dos erros a menor
possível:

$$\text{valor pago} = a + b \times \text{valor financiado} + \text{erro}$$

O coeficiente $b$ é **quanto o valor pago muda, em média, quando o valor
financiado aumenta uma unidade**.

In [ ]:
import statsmodels.formula.api as smf

modelo = smf.ols("valor_pago ~ valor_financiado", data=pagos).fit()
print(modelo.summary().tables[1])
print(f"R² = {modelo.rsquared:.3f}")

In [ ]:
b = modelo.params["valor_financiado"]
baixo, alto = modelo.conf_int().loc["valor_financiado"]
print(f"a cada R$ 1.000 a mais financiados, R$ {b*1000:.0f} a mais pagos "
      f"(IC 95%: R$ {baixo*1000:.0f} a R$ {alto*1000:.0f})")

In [ ]:
(ggplot(pagos, aes(x="valor_financiado", y="valor_pago"))
 + geom_point(alpha=0.6, color="#1f6fb4")
 + geom_smooth(method="lm", color="#e50505")
 + labs(x="valor financiado (R$)", y="valor pago no processo (R$)")
 + theme_minimal())

**Como relatar:** nos 90 processos com pagamento, cada R$ 1.000 a mais de valor
financiado está associado a cerca de **R$ 91 a mais pagos** (IC 95%: R$ 38 a
R$ 144). O valor financiado explica 12% da variação do valor pago: a maior parte
da variação vem de outras coisas.

[Volta ao Índice](#indice)

___
<div id="causa"></div>

# Correlação não é causa

A taxa de juros dos contratos e o ano em que a ação foi ajuizada também andam
juntos.

In [ ]:
base = processos.dropna(subset=["taxa_contrato_pct", "ano_ajuizamento"])
print(base.groupby("ano_ajuizamento")["taxa_contrato_pct"].agg(["mean", "count"]).round(2))
m = smf.ols("taxa_contrato_pct ~ ano_ajuizamento", data=base).fit()
print(f"\nr = {base[['taxa_contrato_pct', 'ano_ajuizamento']].corr().iloc[0, 1]:.2f}   "
      f"b = {m.params['ano_ajuizamento']:.3f} ponto percentual ao mês por ano")

Ninguém diria que o **ano** do ajuizamento causa a taxa. Contratos de ações mais
recentes foram assinados num período de juros de mercado mais altos, e é o juro
de mercado que move as duas coisas. A regressão mede a associação; o motivo
exige conhecer o contexto ou um desenho de pesquisa.

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# Exercício

1. Faça a dispersão, calcule a correlação e ajuste a regressão de
   **`valor_causa`** (extraído da inicial por expressão regular) contra
   **`valor_financiado`** (da planilha). Interprete o coeficiente com a unidade.
2. Compare **`taxa_citada_pct`**, a primeira taxa ao mês que a expressão regular
   achou na petição, com **`taxa_contrato_pct`**, a taxa do contrato na planilha.
   Se a expressão regular pegasse a taxa do contrato, como deveria ser o gráfico?
   Como ele é?

In [ ]:
# item 1

In [ ]:
# item 2

[Volta ao Índice](#indice)